## Ingesting races file

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/2.bronze_helpers

In [0]:
dbutils.widgets.text("p_batch_id", "")  
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/races.csv'
table_name = f'{catalog_name}.{bronze_schema}.races'

### Reading from the races source file

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

races_df_scehma = StructType(
    [
        StructField("season", IntegerType()),
        StructField("round", IntegerType()),
        StructField("url", StringType()),
        StructField("raceName", StringType()),
        StructField("date", DateType()),
        StructField("circuitId", StringType())
    ]
)

In [0]:
races_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(races_df_scehma)
    .load(source_file)
)

In [0]:
display(races_df)


### Metadata Ingestion

In [0]:
races_df_final = add_file_metadata(races_df)

In [0]:
display(races_df_final)

### Writing into the bronze delta table

In [0]:
write_to_bronze(races_df_final, table_name, v_batch_id)

In [0]:
display(spark.table(table_name))